In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Dokładna i zaszumiona symulacja z prymitywami Qiskit Aer

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>

[Dokładna symulacja z prymitywami Qiskit](/guides/simulate-with-qiskit-sdk-primitives) pokazuje, jak używać referencyjnych prymitywów dołączonych do Qiskit do wykonywania dokładnej symulacji Circuit kwantowych. Istniejące procesory kwantowe są podatne na błędy, zwane szumem, więc wyniki dokładnej symulacji niekoniecznie odzwierciedlają wyniki uzyskiwane podczas uruchamiania Circuit na prawdziwym sprzęcie. Podczas gdy referencyjne prymitywy w Qiskit nie obsługują modelowania szumu, [Qiskit Aer](https://qiskit.org/ecosystem/aer/) zawiera implementacje prymitywów obsługujące modelowanie szumu. Qiskit Aer to wysokowydajny symulator Circuit kwantowych, którego możesz używać zamiast referencyjnych prymitywów dla lepszej wydajności i większej liczby funkcji. Jest częścią [ekosystemu Qiskit](https://qiskit.github.io/ecosystem/). W tym artykule demonstrujemy użycie prymitywów Qiskit Aer do dokładnej i zaszumionej symulacji.

> **Note:** - Wymagana jest wersja `qiskit-aer` v0.14 lub nowsza.
> - Choć prymitywy Qiskit Aer implementują interfejsy prymitywów, nie udostępniają tych samych opcji co prymitywy Qiskit Runtime. Na przykład poziom odporności nie jest dostępny w prymitywach Qiskit Aer.
> - Szczegóły dotyczące opcji metod symulacji obsługiwanych przez Aer znajdziesz w [dokumentacji AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator).

Aby zbadać dokładną i zaszumioną symulację, utwórzmy przykładowy Circuit na ośmiu Qubitach:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

Ten Circuit zawiera parametry reprezentujące kąty obrotu dla Gate $R_y$ i $R_z$. Przy symulowaniu tego Circuit musimy podać konkretne wartości dla tych parametrów. W następnej komórce określamy pewne wartości tych parametrów i używamy prymitywu Estimator z Qiskit Aer do obliczenia dokładnej wartości oczekiwanej obserwowalnej $ZZ \cdots Z$.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

Teraz zainicjujemy model szumu uwzględniający błąd depolaryzacyjny o wartości 2% dla każdej Gate CX. W praktyce błędy wynikające z Gate dwuQubitowych, którymi są tutaj Gate CX, są dominującym źródłem błędów przy uruchamianiu Circuit. Informacje na temat konstruowania modeli szumu w Qiskit Aer znajdziesz w artykule [Budowanie modeli szumu](./build-noise-models).

W następnej komórce konstruujemy Estimator uwzględniający ten model szumu i używamy go do obliczenia wartości oczekiwanej obserwowalnej.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

Jak widać, wartość oczekiwana w obecności szumu znacznie odbiega od poprawnej wartości. W praktyce można stosować różne techniki mitygacji błędów, aby przeciwdziałać efektom szumu, jednak dyskusja na ich temat wykracza poza zakres tego artykułu.

Aby uzyskać bardzo przybliżone pojęcie o tym, jak szum wpływa na wynik końcowy, rozważmy nasz model szumu, który dodaje błąd depolaryzacyjny o wartości 2% do każdej Gate CX. Błąd depolaryzacyjny z prawdopodobieństwem $p$ jest zdefiniowany jako kanał kwantowy $E$ o następującym działaniu na macierzy gęstości $\rho$:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

gdzie $n$ to liczba Qubitów, w tym przypadku 2. Oznacza to, że z prawdopodobieństwem $p$ stan jest zastępowany stanem całkowicie mieszanym, a z prawdopodobieństwem $1 - p$ stan jest zachowywany. Po $m$ zastosowaniach kanału depolaryzacyjnego prawdopodobieństwo zachowania stanu wyniosłoby $(1 - p)^m$. Dlatego spodziewamy się, że prawdopodobieństwo zachowania poprawnego stanu na końcu symulacji maleje wykładniczo wraz z liczbą Gate CX w naszym Circuit.

Policzmy liczbę Gate CX w naszym Circuit i obliczmy $(1 - p)^m$. Wywołujemy `count_ops`, aby uzyskać słownik mapujący nazwy Gate na liczniki, i pobieramy wpis dla Gate CX.